# WildSpeech M8.2 — Multiview Pairwise Consensus & Outcome-Adjusted Student Discovery

**One-click GitHub source → cached M8.1 features → independent multi-view clustering → 146,070 pairs → stability and confounder audit → outcome-residual analysis → optional OOF predictive test → 12+ figures and a reproducible report.**

Research safety: score labels are **not** used to define clusters or build the consensus matrix; residuals are computed out of fold, and the predictor trains supervised components exclusively within each training partition. School IDs are used only for validation and confounder analysis. Audio is not reprocessed here.

Start in `SMOKE` to verify paths, then set `MODE='FULL'` for the complete research run. **SMOKE produces incomplete consensus, not a research conclusion.**

## 0 — GitHub-first setup and dependencies

In [ ]:
from pathlib import Path
import os,sys,subprocess,importlib
REPO = 'https://github.com/mybenkhadda/WildSpeech.git'
REF = 'main'
REPO_ROOT = Path('/content/WildSpeech_M82_git') if Path('/content').exists() else Path.cwd()
if Path('/content').exists():
    if not (REPO_ROOT/'.git').exists():
        if REPO_ROOT.exists():
            raise RuntimeError(f'{REPO_ROOT} exists but is not a repository; choose another path')
        subprocess.check_call(['git','clone','--depth','1','--branch',REF,REPO,str(REPO_ROOT)])
    else:
        subprocess.check_call(['git','-C',str(REPO_ROOT),'pull','--ff-only','origin',REF])
PROJECT_ROOT = REPO_ROOT/'WildSpeech_M8'
if not (PROJECT_ROOT/'src'/'wildspeech_m8'/'m82_pipeline.py').is_file():
    raise FileNotFoundError(f'M8.2 modules missing at {PROJECT_ROOT}. Verify main includes the M8.2 commit.')
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',str(PROJECT_ROOT)+'[test]'])
src=str((PROJECT_ROOT/'src').resolve())
if src not in sys.path:sys.path.insert(0,src)
importlib.invalidate_caches()
import wildspeech_m8
print('Installed version:',wildspeech_m8.__version__,'source:',wildspeech_m8.__file__)
assert wildspeech_m8.__version__>='0.8.2'


## 1 — Mount Drive and select experiment settings

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local runtime; Google Drive mount not needed')

DATA_ROOT=Path(os.environ.get('WILDSPEECH_DATA_ROOT','/content/drive/MyDrive/WildSpeech'))
OUTPUT_ROOT=Path(os.environ.get('M8_OUTPUT_ROOT','/content/drive/MyDrive/WildSpeech_M8'))
PROFILE_CSV=os.environ.get('M71_PROFILE_CSV','')
MODE='SMOKE' # Change to 'FULL' after the quick check.
RESAMPLES=5 # Full: runs per combination of cluster count and algorithm; 2 in smoke.
MAX_PER_VIEW=36
RUN_PREDICTIVE=True # Adds nested, training-only OOF residual comparison (extra compute).
print('data:',DATA_ROOT,'outputs:',OUTPUT_ROOT,'mode:',MODE,'profile:',PROFILE_CSV or 'auto-discover')
assert (DATA_ROOT/'train_student_metadata.csv').is_file(),f'Missing official metadata: {DATA_ROOT}'


## 2 — Validate feature bank without re-extracting audio

In [ ]:
from wildspeech_m8.m8_pipeline import Config,prepare_dataset,discover_profile
from wildspeech_m8.m82_views import ViewBuilder
import pandas as pd, numpy as np
from IPython.display import display,Image,Markdown,HTML
cfg_m81=Config(data_root=str(DATA_ROOT),output_root=str(OUTPUT_ROOT),profile_csv=PROFILE_CSV,
               quick=MODE=='SMOKE',max_features=450)
frame,test_frame,y,schools,submission,clips,manifest=prepare_dataset(cfg_m81)
print('Students:',len(frame),'test:',len(test_frame),'features:',frame.shape[1],
      'schools:',len(np.unique(schools)),'possible pairs:',len(frame)*(len(frame)-1)//2)
print('M7 profile:',discover_profile(cfg_m81))
display(manifest.groupby('family').agg(features=('feature','size'),missing=('train_missing','mean')))
print('Cached audio:',(OUTPUT_ROOT/'tables'/'audio_student_features.csv').exists())
print('Cached phonetics:',(OUTPUT_ROOT/'tables'/'phonetic_student_features.csv').exists())


## 3 — Inspect independent views, coverage, and excluded nuisance variables

In [ ]:
builder=ViewBuilder(max_per_view=MAX_PER_VIEW).fit(frame)
views=builder.transform(frame,include_nuisance=True)
for name,v in views.items():
    print(f'{name}: {len(v.columns)} columns, {100*v.observed.mean():.1f}% observed')
    print('   '+', '.join(v.columns[:8]))
print('NOTE: nuisance_audit is NEVER used to construct the student consensus.')
assert len([v for v in views if v!='nuisance_audit'])>=2,'Insufficient independent feature families'


## 4 — Run the full M8.2 consensus and analysis experiment

In [ ]:
from wildspeech_m8.m82_pipeline import M82Config,run_m82
run_cfg=M82Config(data_root=str(DATA_ROOT),output_root=str(OUTPUT_ROOT),profile_csv=PROFILE_CSV,
                  mode=MODE,max_features=450,max_per_view=MAX_PER_VIEW,
                  resamples=RESAMPLES,run_predictive=RUN_PREDICTIVE,
                  predictive_folds=5,predictive_repeats=1,seed=2026)
summary=run_m82(run_cfg)
OUT=OUTPUT_ROOT/'m82';T=OUT/'tables';F=OUT/'figures'
print(summary)


## 5 — Pair category statistics and view coverage

In [ ]:
display(pd.read_csv(T/'pair_category_summary.csv').round(4))
display(pd.read_csv(T/'view_ensemble_summary.csv').round(4))
display(pd.read_csv(T/'ability_matched_pair_summary.csv').round(4))
for name in ('M82_03_consensus_vs_chance.png','M82_04_pair_categories.png','M82_05_pair_coverage.png'):
    if (F/name).is_file():display(Image(filename=str(F/name)))


## 6 — Global coassociation heatmap and one heatmap per feature family

In [ ]:
for name in ['M82_01_consensus_heatmap.png']+sorted([p.name for p in F.glob('M82_02_view_*.png')]):
    if (F/name).is_file():
        print(name)
        display(Image(filename=str(F/name)))


## 7 — Stable student relationships and outcome-discordant behavioral twins

In [ ]:
for name in ('stable_but_outcome_discordant.csv','view_disagreement_pairs.csv'):
    path=T/name
    if path.is_file():
        df=pd.read_csv(path,dtype={'subject_a':str,'subject_b':str})
        print(name,len(df),'pairs')
        wanted=['subject_a','subject_b','category','consensus','chance_expected','view_disagreement','score_gap','screening_residual_gap','same_school']
        display(df[[col for col in wanted if col in df]].head(25))
for name in ('M82_07_consensus_score_gap.png','M82_08_consensus_screening_residual_gap.png','M82_09_pair_residual_gap.png'):
    if (F/name).is_file():display(Image(filename=str(F/name)))


## 8 — Cross-view disagreement: agreement and recording/school confounding

In [ ]:
display(pd.read_csv(T/'school_confound_summary.csv').round(4))
display(pd.read_csv(T/'consensus_score_relationship.csv').round(4))
for name in ('M82_06_view_agreement.png','M82_10_school_pair_confound.png','M82_11_stable_graph_components.png'):
    if (F/name).is_file():display(Image(filename=str(F/name)))


## 9 — Optional inductive predictive validation

In [ ]:
path=T/'pairwise_predictive_fold_metrics.csv'
if path.exists():
    z=pd.read_csv(path)
    display(z.groupby('model')[['r2','mae','rmse']].agg(['mean','std']).round(5))
    display(pd.read_csv(T/'pairwise_predictive_student_bootstrap.csv'))
    display(pd.read_csv(T/'pairwise_predictive_school_bootstrap.csv'))
    if (F/'M82_12_predictive_cv.png').exists():display(Image(filename=str(F/'M82_12_predictive_cv.png')))
else:
    print('Prediction not run. Change RUN_PREDICTIVE=True and rerun §4.')


## 10 — Read the automatic evidence report and open the figure atlas

In [ ]:
report=OUT/'M82_RESEARCH_SUMMARY.md'
if report.exists():display(Markdown(report.read_text()))
print('Figure atlas:',OUT/'M82_RESEARCH_ATLAS.html')
print('Full pair table:',T/'all_student_pairs.parquet' if (T/'all_student_pairs.parquet').exists() else T/'all_student_pairs.csv.gz')
print('Generated figures:',len(list(F.glob('M82*.png'))))


## 11 — How to interpret the findings

- **Stable together** means reproducible co-clustering in the tested feature views; it is **not** proof of a biological or educational type.
- **Stable apart** is chance-adjusted; the base chance depends on actual cluster size and cluster count.
- **Low coverage** is not similarity. Two missing values must not create a false stable pair.
- Pair-level Spearman and gap distributions are descriptive; student pairs are not independent.
- Identical validation folds are used for global and structural-prototype residual models.
- Group outcome differences are **not** used to build the unsupervised consensus; they are analyzed afterward.
- If stable pairs are mostly explained by school or recording conditions, remove those signals and repeat.
- Do not use the most promising pair groups to select a leaderboard model without further independent validation.
- `SMOKE` outputs are only pipeline diagnostics. For scientific interpretation use `FULL` on the complete M8.1 feature bank.